# 01 · Eingaben für Spatial Hybrid v1 vorbereiten

Aus Rohtracking und gespeicherten Vorstufen entstehen Physics v3 und die räumliche Distanzdifferenz. Die NPZ ist die Eingabe für Notebook 02. Es wird kein Modell trainiert; Heuristik-, AdaBoost-/OOF-Vorhersagen und Bewegungsfeatures stammen aus dem bisherigen Entwicklungsstand.

[Notebook 02 →](../02_spatial_hybrid_v1/02_spatial_hybrid_v1.ipynb)

In [ ]:
from pathlib import Path
import json
import sys
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from IPython.display import display

## 1. Vorstufen laden

Die gespeicherten OOF-Vorhersagen ersetzen im ursprünglichen öffentlichen Abschnitt die Full-fit-Statusvorhersagen. Das übernimmt die historische Berechnung; neue Zielwerte werden nicht gelesen.

In [ ]:
PROJECT_DIR = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "reference/mtah2027_official_3de6b60.zip").is_file()
    )
)
N_FRAMES, FPS = (199319, 25)
RANDOM_STATE = 42
HALVES = [("1. Halbzeit", 11185, 90651), ("2. Halbzeit", 110808, 190197)]
paths = {
    "contact_04": PROJECT_DIR
    / "artifacts/upstream/contact_classification/submissions/contact_state_shared_D.csv",
    "status_06": PROJECT_DIR
    / "artifacts/upstream/stoppage_detection/submissions/contact_state_stoppage_v1.csv",
    "ml_08": PROJECT_DIR / "artifacts/upstream/player_status_ml/full_match_status_diagnostics.csv",
    "oof_08": PROJECT_DIR / "artifacts/upstream/player_status_ml/public_oof_predictions.csv",
    "movement": PROJECT_DIR / "artifacts/upstream/movement_features.csv",
    "contact_config": PROJECT_DIR / "artifacts/upstream/contact_classification/run_manifest.json",
    "raw_tracking": PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.txt",
    "metadata": PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.xml",
    "physics_expected": PROJECT_DIR / "artifacts/spatial_hybrid_v1/references/physics_v3_reference.csv",
    "hybrid_expected": PROJECT_DIR / "artifacts/spatial_hybrid_v1/references/spatial_hybrid_v1.csv",
    "exporter": PROJECT_DIR / "main approach/export_spatial_hybrid_inputs.py",
}
contact_04 = pd.read_csv(paths["contact_04"])
status_06 = pd.read_csv(paths["status_06"])
ml_08 = pd.read_csv(paths["ml_08"], usecols=["frame", "AdaBoost_median91"])
oof = pd.read_csv(paths["oof_08"], usecols=["frame", "AdaBoost_median91"])
movement_features = pd.read_csv(paths["movement"])
base_labels = status_06["label"].to_numpy().copy()
contact_labels = contact_04["label"].to_numpy()
ada_labels = ml_08["AdaBoost_median91"].to_numpy().copy()
oof_frames = oof["frame"].to_numpy(dtype=int)
ada_labels[oof_frames] = oof["AdaBoost_median91"].to_numpy()
print("Python:", sys.version.split()[0], "NumPy:", np.__version__, "pandas:", pd.__version__)
display(movement_features.head())

## 2. Tracking und Kontaktgeometrie

Die XML-Datei ordnet Track-Slots den Teams zu. Koordinaten werden in Meter umgerechnet; RTS-Glättung auf vollständigen Ballruns liefert Beschleunigung und den nächsten Spielerabstand. Die Kontaktparameter stammen aus dem bestehenden Manifest.

In [ ]:
CONTACT_DIR = PROJECT_DIR / "artifacts/upstream/contact_classification"
config = json.loads((CONTACT_DIR / "run_manifest.json").read_text())
PITCH = np.asarray(config["pitch_m"], dtype=float)
NEAR_DISTANCE_M = config["near_distance_m"]
PEAK_ACCELERATION_M_S2 = config["peak_acceleration_m_s2"]
MIN_RUN_SECONDS = config["min_run_seconds"]
EDGE_GUARD_SECONDS = config["edge_guard_seconds"]
MEASUREMENT_SIGMA_M = config["measurement_sigma_m"]
PROCESS_SIGMA_M_S2 = config["process_sigma_m_s2"]
META_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.xml"
RAW_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.txt"
root = ET.parse(META_PATH).getroot()
team_nodes = root.findall("./Metadata/Teams/Team")
team_map = {team.get("id"): "H" if i == 0 else "A" for i, team in enumerate(team_nodes)}
channel_to_track = {
    channel.get("id"): channel.get("playerId")
    for channel in root.findall("./Metadata/PlayerChannels/PlayerChannel")
}
spec = root.find("./DataFormatSpecifications/DataFormatSpecification")
track_ids = [
    channel_to_track[group.find("PlayerChannelRef").get("playerChannelId")]
    for group in spec.findall("./SplitRegister/SplitRegister")
]
player_team = {
    player.get("id"): team_map[player.get("teamId")] for player in root.findall("./Metadata/Players/Player")
}
track_sides = np.asarray([player_team[track_id] for track_id in track_ids])
positions = np.empty((N_FRAMES, len(track_ids) + 1, 2), dtype=float)
with RAW_PATH.open() as handle:
    for frame, line in enumerate(handle):
        raw_frame, players_raw, ball_raw = line.strip().split(":")
        values = np.fromstring(players_raw.replace(";", ",") + "," + ball_raw, sep=",").reshape(-1, 3)
        positions[frame] = values[:, :2] * PITCH
player_xy_contact = positions[:, :-1]
ball_xy_contact = positions[:, -1]
ball_seen = np.isfinite(ball_xy_contact).all(axis=1)


def smooth_ball_rts(points, fps, measurement_sigma, process_sigma):
    dt = 1 / fps
    transition = np.eye(4)
    transition[0, 2] = dt
    transition[1, 3] = dt
    observation = np.eye(2, 4)
    acceleration_map = np.r_[np.eye(2) * dt**2 / 2, np.eye(2) * dt]
    process_noise = acceleration_map @ acceleration_map.T * process_sigma**2
    measurement_noise = np.eye(2) * measurement_sigma**2
    state = np.r_[points[0], (points[1] - points[0]) / dt]
    covariance = np.diag([measurement_sigma**2, measurement_sigma**2, 100.0, 100.0])
    predicted = np.empty((len(points), 4))
    filtered = np.empty_like(predicted)
    predicted_cov = np.empty((len(points), 4, 4))
    filtered_cov = np.empty_like(predicted_cov)
    for i, point in enumerate(points):
        if i:
            state = transition @ state
            covariance = transition @ covariance @ transition.T + process_noise
        predicted[i] = state
        predicted_cov[i] = covariance
        gain = np.linalg.solve(
            observation @ covariance @ observation.T + measurement_noise, (covariance @ observation.T).T
        ).T
        state = state + gain @ (point - observation @ state)
        residual_map = np.eye(4) - gain @ observation
        covariance = residual_map @ covariance @ residual_map.T + gain @ measurement_noise @ gain.T
        filtered[i] = state
        filtered_cov[i] = covariance
    smoothed = filtered.copy()
    for i in range(len(points) - 2, -1, -1):
        gain = np.linalg.solve(predicted_cov[i + 1].T, (filtered_cov[i] @ transition.T).T).T
        smoothed[i] += gain @ (smoothed[i + 1] - predicted[i + 1])
    return smoothed


starts = np.flatnonzero(ball_seen & ~np.r_[False, ball_seen[:-1]])
ends = np.flatnonzero(ball_seen & ~np.r_[ball_seen[1:], False])
acceleration = np.full(N_FRAMES, np.nan)
nearest_distance = np.full(N_FRAMES, np.nan)
nearest_offset = np.full(N_FRAMES, np.nan)
eligible = np.zeros(N_FRAMES, dtype=bool)
edge_guard = round(EDGE_GUARD_SECONDS * FPS)
for first, last in zip(starts, ends):
    idx = np.arange(first, last + 1)
    if len(idx) / FPS < MIN_RUN_SECONDS:
        continue
    state = smooth_ball_rts(ball_xy_contact[idx], FPS, MEASUREMENT_SIGMA_M, PROCESS_SIGMA_M_S2)
    acceleration[idx] = np.linalg.norm(np.gradient(state[:, 2:], 1 / FPS, axis=0), axis=1)
    distances = np.linalg.norm(player_xy_contact[idx] - state[:, None, :2], axis=2)
    distances = np.where(np.isfinite(distances), distances, np.inf)
    distance_min = distances.min(axis=1)
    offset_min = distances.argmin(axis=1).astype(float)
    offset_min[~np.isfinite(distance_min)] = np.nan
    distance_min[~np.isfinite(distance_min)] = np.nan
    nearest_distance[idx] = distance_min
    nearest_offset[idx] = offset_min
    if len(idx) > 2 * edge_guard:
        eligible[idx[edge_guard:-edge_guard]] = True
acc_series = pd.Series(acceleration)
local_peak = (acc_series.gt(acc_series.shift()) & acc_series.ge(acc_series.shift(-1))).to_numpy()
candidate_mask = (
    eligible & local_peak & (acceleration >= PEAK_ACCELERATION_M_S2) & (nearest_distance <= NEAR_DISTANCE_M)
)
contact_candidate_side = np.full(N_FRAMES, "", dtype="<U1")
contact_candidate_side[candidate_mask] = track_sides[nearest_offset[candidate_mask].astype(int)]
print("Kontaktkandidaten:", int(candidate_mask.sum()))

## 3. Status- und Besitzdecoder

Die bestehende Kombination aus AdaBoost und Heuristik liefert D/L-Evidenz für Viterbi. Innerhalb von LIVE haben die Heuristiklabels Vorrang, Kontaktlabels dienen als Ergänzung; Besitz wird innerhalb der Segmente fortgeschrieben.

In [ ]:
heur_d = base_labels == "D"
ada_d = ada_labels == "D"
ada_valid = np.isin(ada_labels, ["D", "L"])
p_d = np.full(N_FRAMES, 0.04, dtype=float)
p_d[~ada_valid & heur_d] = 0.9
p_d[~ada_valid & ~heur_d] = 0.1
p_d[ada_valid & heur_d & ada_d] = 0.96
p_d[ada_valid & heur_d & ~ada_d] = 0.7
p_d[ada_valid & ~heur_d & ada_d] = 0.62
p_d[ada_valid & ~heur_d & ~ada_d] = 0.04
base_logit = np.log(p_d / (1 - p_d))


def viterbi_status(logit_d, switch_penalty=4.0):
    """
    Zwei Zustände:
      0 = LIVE
      1 = D

    Maximiert die gesamte Evidenz über die Zeit
    und bestraft häufige Zustandswechsel.

    switch_penalty ist ein Startparameter
    (keine auf GT trainierte Größe).
    """
    logit_d = np.asarray(logit_d, dtype=float)
    n = len(logit_d)
    logp_d = -np.logaddexp(0, -logit_d)
    logp_l = -np.logaddexp(0, logit_d)
    back = np.zeros((n, 2), dtype=np.uint8)
    score_l = logp_l[0]
    score_d = logp_d[0]
    for t in range(1, n):
        from_l = score_l
        from_d = score_d - switch_penalty
        if from_l >= from_d:
            new_l = from_l + logp_l[t]
            back[t, 0] = 0
        else:
            new_l = from_d + logp_l[t]
            back[t, 0] = 1
        from_l = score_l - switch_penalty
        from_d = score_d
        if from_d >= from_l:
            new_d = from_d + logp_d[t]
            back[t, 1] = 1
        else:
            new_d = from_l + logp_d[t]
            back[t, 1] = 0
        score_l, score_d = (new_l, new_d)
    states = np.zeros(n, dtype=np.uint8)
    states[-1] = 1 if score_d > score_l else 0
    for t in range(n - 1, 0, -1):
        states[t - 1] = back[t, states[t]]
    return np.where(states == 1, "D", "L")


def reconstruct_possession(status_labels, base_labels, contact_labels):
    """
    Hierarchische Rekonstruktion:

    Stufe 1: LIVE oder D.
    Stufe 2: H/A-Evidenz und zeitliche Besitzfortführung.

    Keine Ground-Truth-Informationen werden verwendet.
    """
    status_labels = np.asarray(status_labels)
    base_labels = np.asarray(base_labels)
    contact_labels = np.asarray(contact_labels)
    n = len(status_labels)
    live = status_labels == "L"
    base_valid = np.isin(base_labels, ["H", "A"])
    contact_valid = np.isin(contact_labels, ["H", "A"])
    evidence = np.full(n, None, dtype=object)
    evidence[contact_valid] = contact_labels[contact_valid]
    evidence[base_valid] = base_labels[base_valid]
    evidence[~live] = None
    segment_id = np.cumsum(np.r_[True, live[1:] != live[:-1]])
    evidence_series = pd.Series(evidence)
    possession = evidence_series.groupby(segment_id).ffill()
    possession = possession.groupby(segment_id).bfill()
    labels = np.full(n, "D", dtype="<U1")
    reconstructed = possession.to_numpy()
    known = live & pd.notna(reconstructed)
    unknown = live & ~known
    labels[known] = reconstructed[known]
    labels[unknown] = "X"
    return (
        labels,
        {
            "live_frames": int(live.sum()),
            "reconstructed_H": int((labels == "H").sum()),
            "reconstructed_A": int((labels == "A").sum()),
            "unresolved_X": int(unknown.sum()),
        },
    )

## 4. Physics v3 berechnen

Spielerbewegung, Ballgeschwindigkeit und Nähe schwächen widersprüchliche D-Evidenz ab. Bei fehlendem Ball kommt die bestehende Bewegungsregel hinzu. Gewichte und Schwellen bleiben unverändert.

In [ ]:
moving = movement_features["moving_fraction"].to_numpy(float)
ball_speed = movement_features["ball_speed_mps"].to_numpy(float)
ball_coverage = movement_features["ball_coverage"].to_numpy(float)

nearest_smooth = pd.Series(nearest_distance).rolling(FPS, center=True, min_periods=5).median().to_numpy()
heur_d = base_labels == "D"
ada_d = ada_labels == "D"
ada_known = np.isin(ada_labels, ["D", "L"])
disagreement = ada_known & (heur_d != ada_d)
active = np.zeros(N_FRAMES, dtype=bool)
for _, start, end in HALVES:
    active[start : end + 1] = True

physical_live = (
    active
    & disagreement
    & np.isfinite(moving)
    & np.isfinite(ball_speed)
    & np.isfinite(nearest_smooth)
    & (moving >= 0.4)
    & (ball_speed >= 6.0)
    & (ball_coverage >= 0.5)
    & (nearest_smooth <= 3.5)
)
PHYSICS_WEIGHT = 1.5

no_ball_live = (
    active
    & (base_labels == "D")
    & (ada_labels == "L")
    & np.isfinite(moving)
    & np.isfinite(ball_coverage)
    & (ball_coverage <= 0.2)
    & (moving >= 0.6)
)
NO_BALL_WEIGHT = 1.5

combined_logit = base_logit.copy()
combined_logit[physical_live] -= PHYSICS_WEIGHT
combined_logit[no_ball_live] -= NO_BALL_WEIGHT
status_physics_v3 = viterbi_status(combined_logit / FPS, switch_penalty=1.0)
hier_physics_v3, diag_v3 = reconstruct_possession(status_physics_v3, base_labels, contact_labels)

## 5. Räumliche Evidenz berechnen

Die Minimaldistanz zum Ball wird je Team blockweise aus den ursprünglichen Koordinaten berechnet. `d_A − d_H` ist positiv, wenn H näher ist. Fehlende Ballbeobachtungen bleiben fehlende Evidenz.

In [ ]:
def resolve_team(raw_team):
    """Team-ID anhand expliziter Metadaten H/A zuordnen."""
    aliases = [str(raw_team)]
    for key, value in team_map.items():
        if str(key) == str(raw_team):
            aliases.append(str(value))
        if str(value) == str(raw_team):
            aliases.append(str(key))
    for alias in aliases:
        a = alias.strip().lower()
        if a in ("h", "home", "bar", "barca", "barça"):
            return "H"
        if "barcelona" in a:
            return "H"
        if a in ("a", "away", "ath", "bilbao"):
            return "A"
        if "athletic" in a:
            return "A"
    return None


track_rows = []
for track_number in range(1, 47):
    raw_team = player_team.get(str(track_number), player_team.get(track_number))
    track_rows.append(
        {
            "track_number": track_number,
            "array_index": track_number - 1,
            "raw_team": raw_team,
            "side": resolve_team(raw_team),
        }
    )
df_track_teams = pd.DataFrame(track_rows)
print("\nSpieler-Tracks nach Team:")
display(df_track_teams.groupby(["raw_team", "side"], dropna=False).size().reset_index(name="tracks"))
if df_track_teams["side"].isna().any():
    raise ValueError("Teamzuordnung nicht eindeutig. Bitte die angezeigte Team-Mapping-Tabelle prüfen.")
idx_H = df_track_teams.loc[df_track_teams["side"] == "H", "array_index"].to_numpy(dtype=int)
idx_A = df_track_teams.loc[df_track_teams["side"] == "A", "array_index"].to_numpy(dtype=int)
ball_valid_mask = np.asarray(ball_seen, dtype=bool)


def team_min_distance(track_indices, chunk_size=10000):
    result = np.full(N_FRAMES, np.nan, dtype=np.float32)
    for start in range(0, N_FRAMES, chunk_size):
        end = min(N_FRAMES, start + chunk_size)
        p = np.asarray(player_xy_contact[start:end][:, track_indices, :], dtype=np.float32)
        b = np.asarray(ball_xy_contact[start:end], dtype=np.float32)
        distances = np.hypot(p[:, :, 0] - b[:, None, 0], p[:, :, 1] - b[:, None, 1])
        distances = np.where(np.isfinite(distances), distances, np.inf)
        nearest = distances.min(axis=1)
        nearest[~np.isfinite(nearest)] = np.nan
        result[start:end] = nearest
    result[~ball_valid_mask] = np.nan
    return result


distance_H = team_min_distance(idx_H)
distance_A = team_min_distance(idx_A)
proximity_margin_H = distance_A - distance_H

## 6. Eingaben für Notebook 02 speichern

Die bisherige Hybridvorhersage wird als Vergleichswert mitgegeben. Sie beeinflusst die Berechnung in Notebook 02 nicht. Ein vorhandener NPZ-Export bleibt erhalten.

In [ ]:
DESTINATION = PROJECT_DIR / "artifacts/spatial_hybrid_v1/inputs/spatial_hybrid_inputs_v1.npz"
export_values = {
    "reference": np.asarray(hier_physics_v3, dtype="<U1"),
    "status": np.asarray(status_physics_v3, dtype="<U1"),
    "proximity_margin_H": np.asarray(proximity_margin_H, dtype=np.float32),
    "expected_hybrid": pd.read_csv(paths["hybrid_expected"])["label"].to_numpy(dtype="<U1"),
    "halves": np.asarray([(a, b) for _, a, b in HALVES], dtype=np.int64),
    "n_frames": np.array(N_FRAMES, dtype=np.int64),
    "fps": np.array(FPS, dtype=np.int64),
}
if not DESTINATION.exists():
    DESTINATION.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(DESTINATION, **export_values)
print("Eingaben für Notebook 02:", DESTINATION)
display(pd.Series(hier_physics_v3).value_counts().rename("Physics-v3-Frames").to_frame())

## Ergebnis

Physics v3 und die Distanzdifferenz sind vorbereitet. Weiter mit Notebook 02. Die Berechnung verwendet gespeicherte Vorstufen und ist damit noch kein vollständiger Neuaufbau allein aus Rohdaten.